# IAD Pipeline
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

## 1. Environment Setup
Configure database URI and API keys.

In [ ]:
import os
import sys
import warnings

# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'


sys.path.append("..")

# Now safe to import
import wandb
import logging
from pathlib import Path
from src.manager import AnomalyDetectionManager as ADM

wandb.login()

## 2. Configuration
Set your run parameters here before executing the pipeline.

In [ ]:
tiling      = True
modelName   = "patchcore"
modelConfig = "patchcore"
modelTrainConfig = f"Training_{modelName}"
datasetName = "Irlbacher2023"
split       = ("train","test")
category    = "MDF6"
tiledEnsembleConfig = "IrlbacherTiledEnsemble.yaml" # "TiledEnsemble.yaml", "IrlbacherTiledEnsemble.yaml"
train       = True
evaluate    = True
datasetDir  = Path("../datasets/")
configDir   = Path("../configs/")
outputPath  = Path("../results/")
# ckptPath    = Path("../results/MVTecADShort/cable/padim/tiled/checkpoints")

## 3. Initialise Logger & IAD

In [ ]:
logger = logging.getLogger("logger")

manager = ADM(configDir=configDir, datasetDir=datasetDir,outputPath=outputPath)

## 4. Load Dataset & Generate Model

In [ ]:
manager.generateModel(f"{modelConfig}.yaml", configDir)
datasetPath = datasetDir/datasetName
manager.loadDatasetFromDisk(datasetPath, datasetName, overwrite=True, merge=False, split=split)
manager.selectCategory(category)
if manager.FO_Dataset is None:
    raise RuntimeError("Dataset failed to load — check your dataset path and name.")
if tiling:
    manager.setupTiling(configDir / "Tiling" / f"{tiledEnsembleConfig}")
manager.adjustOutputPath()
# manager.copyFilesToOutputPath()

## 4.1 Inspect Dataset

In [ ]:
manager.launchSession()

## 5. Training

In [ ]:

warnings.filterwarnings("ignore", category=FutureWarning, module="timm.models.layers")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="openvino.runtime")
if train:
    manager.train(configDir / "Trainer" / Path(f"{modelTrainConfig}.yaml"), tiling=tiling)

In [ ]:
manager.launchSession()

## 6. Evaluation & Prediction

In [ ]:
# import time
# time.sleep(5)
# Database needs to load

if evaluate:
    if manager.ckptPath is not None:
        if not tiling:
            manager.loadCheckpoint(manager.ckptPath, f"{modelName}")
        manager.eval(configDir / "Trainer"/ "Evaluation.yaml", tiling=tiling)
        manager.launchSession()
    else:
        print("No checkpoint found — skipping evaluation.")
    

## 7. Evaluate on single unknown Image

In [ ]:
# ckptPath = manager.ckptPath.parent.resolve()
# print(ckptPath)

In [ ]:

# # ckptPath = manager.ckptPath.parent.resolve()
# # print(ckptPath)
# manager.loadDatasetFromDisk(datasetPath=datasetDir / "MVTecADShortPred", datasetName="cablePred36", split=("pred",), overwrite=True)
# # manager.selectCategory("cable")

# if manager.ckptPath is not None:
#     if not tiling:
#         manager.loadCheckpoint(manager.ckptPath, f"{modelName}")
#     if tiling:
#         manager.setupTiling(configDir / "TiledEnsemblePred.yaml")
#     manager.predict(config=configDir / "Predict.yaml", tiling=tiling, ckptPath=ckptPath)
#     print(manager.FO_Dataset)
#     manager.launchSession()
#     print(manager.FO_Dataset)
#     manager.launchSession()